# 03 · Train plant identity encoders on the packaged 18-class split

Run in Colab after uploading `plant_export_v20261026_2355546_holdout10_seed42.zip` to `MyDrive/AI/meta-global-hackaton/data/plant_export_v20261026_2355546_holdout10_seed42.zip`. The package contains standard `train/`, `val/`, `test/`, and `external_test/` class folders plus an image manifest. This notebook reads those exact packaged images, trains on train/validation, evaluates the internal test split, and leaves the 180-image holdout untouched for notebook 06.

The holdout comes from the **same source archive** as training, so it is an internal holdout rather than an independent real-world test.


## Classes and split policy

The package has 18 alphabetically ordered class folders, with PyTorch targets 0–17. It was built from the original archive after removing three exact duplicate records. Its `external_test` directory has exactly 10 images per class. The source archive's old split and source label index remain in the package manifest for provenance; they do not determine model membership.


In [ ]:
# Colab supplies PyTorch/torchvision; install Hugging Face model support.
import subprocess
import sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.57,<5', 'scikit-learn>=1.5',
])

0

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import copy
import hashlib
import json
import os
import random
import sys

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import EfficientNet_B2_Weights, efficientnet_b2
from transformers import AutoImageProcessor, AutoModel, AutoProcessor
from huggingface_hub import get_token

SEED = 42
EPOCHS = 12
HEAD_BATCH_SIZE = 64
FEATURE_BATCH_SIZE = 8
LEARNING_RATE = 1e-3
MODELS_TO_TRAIN = ('efficientnet_b2', 'dinov3', 'siglip2')
SAVE_TO_DRIVE = True
DATASET_VERSION = 'v20261026_2355546'
PACKAGE_NAME = f'plant_export_{DATASET_VERSION}_holdout10_seed{SEED}'
ARCHIVE_RELATIVE_PATH = Path('AI/meta-global-hackaton/data') / f'{PACKAGE_NAME}.zip'
HF_DINOV3_ID = 'facebook/dinov3-vits16-pretrain-lvd1689m'
HF_SIGLIP2_ID = 'google/siglip2-base-patch16-224'
assert MODELS_TO_TRAIN and set(MODELS_TO_TRAIN) <= {'efficientnet_b2', 'dinov3', 'siglip2'}
assert EPOCHS >= 1

# Source indices and folder names come from the supplied export metadata.
CLASS_INFO = [
    (1, 'aglaonema', 'aglaonema'),
    (2, 'aloe vera', 'aloe-vera'),
    (3, 'chamaedorea elegans', 'chamaedorea-elegans'),
    (4, 'chlorophytum comosum', 'chlorophytum-comosum'),
    (5, 'chrysanthemum', 'chrysanthemum'),
    (6, 'crassula ovata', 'crassula-ovata'),
    (7, 'cyclamen', 'cyclamen'),
    (8, 'dieffenbachia', 'dieffenbachia'),
    (10, 'epipremnum aureum', 'epipremnum-aureum'),
    (11, 'euphorbia pulcherrima', 'euphorbia-pulcherrima'),
    (12, 'hemerocallis', 'hemerocallis'),
    (13, 'monstera deliciosa', 'monstera-deliciosa'),
    (14, 'narcissus', 'narcissus'),
    (15, 'nephrolepis exaltata', 'nephrolepis-exaltata'),
    (16, 'phalaenopsis', 'phalaenopsis'),
    (17, 'philodendron hederaceum', 'philodendron-hederaceum'),
    (18, 'saintpaulia', 'saintpaulia'),
    (19, 'spathiphyllum', 'spathiphyllum'),
]
CLASS_NAMES = [name for _, name, _ in CLASS_INFO]
assert len(CLASS_INFO) == 18 and len(set(CLASS_NAMES)) == 18
assert len({source_index for source_index, _, _ in CLASS_INFO}) == 18

IN_COLAB = 'google.colab' in sys.modules
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Use Colab for Drive storage, or set SAVE_TO_DRIVE=False locally.')
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    BASE_RUNS_DIR = DRIVE_ROOT / 'AI/meta-global-hackaton/runs'
    ARCHIVE_PATH = DRIVE_ROOT / ARCHIVE_RELATIVE_PATH
else:
    BASE_RUNS_DIR = Path('./data/processed/runs').resolve()
    ARCHIVE_PATH = Path('./data/processed') / f'{PACKAGE_NAME}.zip'
EXPERIMENT_ID = PACKAGE_NAME
RUNS_DIR = BASE_RUNS_DIR / EXPERIMENT_ID
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = get_token()
if 'dinov3' in MODELS_TO_TRAIN and not HF_TOKEN:
    raise RuntimeError(
        'DINOv3 is gated. Accept model access and set HF_TOKEN in Colab Secrets, '
        'or omit dinov3 from MODELS_TO_TRAIN.'
    )
if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Dataset archive not found: {ARCHIVE_PATH}')
print('Device:', DEVICE, '| Archive:', ARCHIVE_PATH, '| Runs:', RUNS_DIR)

Mounted at /content/drive
Device: cuda | Archive: /content/drive/MyDrive/AI/meta-global-hackaton/data/plant_export_v20261026_2355546_holdout10_seed42.zip | Runs: /content/drive/MyDrive/AI/meta-global-hackaton/runs/plant_export_v20261026_2355546_holdout10_seed42


## 1. Validate the packaged ZIP and export run manifests

Read `manifest.csv` and class mapping from the package. Verify every packaged image against its SHA-256, class directory, and recorded split. Then copy the four manifest tables into the new Drive run directory. Images remain in the ZIP; no extraction or resizing is needed for this notebook.


In [ ]:
SPLIT_NAMES = ('train', 'val', 'test', 'external_test')
with ZipFile(ARCHIVE_PATH) as archive:
    package_manifest = pd.read_csv(BytesIO(archive.read('manifest.csv')))
    package_metadata = json.loads(archive.read('metadata.json'))
    package_mapping = json.loads(archive.read('class_to_idx.json'))
    required = {'split', 'class_name', 'class_index', 'folder_name', 'file_path',
                'sha256', 'source_zip_member', 'source_split', 'source_label_index'}
    if not required <= set(package_manifest.columns):
        raise ValueError(f'Package manifest lacks: {sorted(required - set(package_manifest.columns))}')
    if package_manifest.file_path.duplicated().any() or package_manifest.sha256.duplicated().any():
        raise ValueError('Packaged image paths or hashes are duplicated')
    if package_metadata['by_split'] != package_manifest.split.value_counts().to_dict():
        raise ValueError('Packaged split counts differ from metadata')
    if package_mapping['index_to_class'] != CLASS_NAMES:
        raise ValueError('Packaged class order differs from model class order')
    expected_folders = package_mapping['class_to_idx']
    for row in package_manifest.itertuples(index=False):
        if row.split not in SPLIT_NAMES:
            raise ValueError(f'Unknown split: {row.split}')
        if Path(row.file_path).parts[:2] != (row.split, row.folder_name):
            raise ValueError(f'Image is in the wrong split/class folder: {row.file_path}')
        if expected_folders[row.folder_name] != row.class_index:
            raise ValueError(f'Class index mismatch: {row.file_path}')
        image_bytes = archive.read(row.file_path)
        if hashlib.sha256(image_bytes).hexdigest() != row.sha256:
            raise ValueError(f'Image hash mismatch: {row.file_path}')
        with Image.open(BytesIO(image_bytes)) as image:
            image.verify()
identity = sorted((row.sha256, row.split, row.class_name)
                  for row in package_manifest.itertuples(index=False))
DATASET_FINGERPRINT = hashlib.sha256(
    json.dumps(identity, separators=(',', ':')).encode()).hexdigest()[:16]
if DATASET_FINGERPRINT != package_metadata['split_fingerprint']:
    raise ValueError('Packaged split fingerprint does not match the manifest')
selected_manifest = package_manifest.rename(columns={
    'class_index': 'target', 'file_path': 'zip_member',
    'source_zip_member': 'image_id',
})
MODEL_SPLITS = {name: selected_manifest[selected_manifest.split == name]
                .sort_values('image_id').reset_index(drop=True)
                for name in SPLIT_NAMES[:3]}
EXTERNAL_SPLIT = selected_manifest[selected_manifest.split == 'external_test']
EXTERNAL_SPLIT = EXTERNAL_SPLIT.sort_values('image_id').reset_index(drop=True)
if len(EXTERNAL_SPLIT) != 180 or not EXTERNAL_SPLIT.groupby('class_name').size().eq(10).all():
    raise ValueError('Internal holdout must have exactly 10 images per class')
selected = pd.concat(MODEL_SPLITS.values(), ignore_index=True)
selected = selected.sort_values('image_id').reset_index(drop=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
for split_name, frame in [*MODEL_SPLITS.items(), ('external_test', EXTERNAL_SPLIT)]:
    frame.to_csv(RUNS_DIR / f'{split_name}.csv', index=False)
counts = pd.DataFrame({name: frame.class_name.value_counts()
                       for name, frame in [*MODEL_SPLITS.items(), ('external_test', EXTERNAL_SPLIT)]})
print('Training/validation/test images:', len(selected), '| Internal holdout:', len(EXTERNAL_SPLIT))
print('Split fingerprint:', DATASET_FINGERPRINT)
display(counts.reindex(CLASS_NAMES).fillna(0).astype(int))


Training/validation/test images: 1112 | Internal holdout: 180
Split fingerprint: a36fa81fded18e69


,train,val,test,external_test
class_name,,,,
aglaonema,60,8,8,10
aloe vera,49,6,6,10
chamaedorea elegans,41,5,5,10
chlorophytum comosum,43,5,5,10
chrysanthemum,64,8,8,10
crassula ovata,52,7,7,10
cyclamen,38,5,5,10
dieffenbachia,50,6,6,10
epipremnum aureum,31,4,4,10


## 2. Extract frozen encoder features

All three encoders read the packaged train/validation/test images directly from the ZIP and apply their own pretrained preprocessing. The 180 held-out images are excluded from feature extraction and training. Encoders run sequentially to fit a free Colab GPU.


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_encoder(model_name: str):
    if model_name == 'efficientnet_b2':
        weights = EfficientNet_B2_Weights.IMAGENET1K_V1
        encoder = efficientnet_b2(weights=weights)
        encoder.classifier = nn.Identity()
        return encoder.to(DEVICE).eval(), weights.transforms(), None
    encoder_id = HF_DINOV3_ID if model_name == 'dinov3' else HF_SIGLIP2_ID
    token = HF_TOKEN if model_name == 'dinov3' else None
    try:
        processor = (AutoImageProcessor if model_name == 'dinov3' else AutoProcessor).from_pretrained(
            encoder_id, token=token
        )
        encoder = AutoModel.from_pretrained(encoder_id, token=token).to(DEVICE).eval()
    except OSError as exc:
        if model_name == 'dinov3':
            raise RuntimeError('DINOv3 access failed. Accept its license and check HF_TOKEN.') from exc
        raise
    return encoder, processor, encoder_id


def embedding_tensor(output):
    if torch.is_tensor(output):
        return output
    for attribute in ('image_embeds', 'pooler_output'):
        value = getattr(output, attribute, None)
        if torch.is_tensor(value):
            return value
    hidden = getattr(output, 'last_hidden_state', None)
    if torch.is_tensor(hidden):
        return hidden[:, 0]
    raise TypeError(f'No image embedding tensor in {type(output).__name__}')


def extract_features(model_name: str) -> torch.Tensor:
    encoder, processor, _ = make_encoder(model_name)
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    chunks = []
    with ZipFile(ARCHIVE_PATH) as archive, torch.inference_mode():
        for start in range(0, len(selected), FEATURE_BATCH_SIZE):
            batch = selected.iloc[start:start + FEATURE_BATCH_SIZE]
            images = []
            for row in batch.itertuples(index=False):
                with Image.open(BytesIO(archive.read(row.zip_member))) as image:
                    images.append(image.convert('RGB'))
            if model_name == 'efficientnet_b2':
                inputs = torch.stack([processor(image) for image in images]).to(DEVICE)
                features = encoder(inputs)
            else:
                inputs = processor(images=images, return_tensors='pt')
                inputs = {key: value.to(DEVICE) if torch.is_tensor(value) else value
                          for key, value in inputs.items()}
                if model_name == 'siglip2':
                    output = encoder.get_image_features(**inputs)
                else:
                    output = encoder(**inputs, return_dict=True)
                features = embedding_tensor(output)
            chunks.append(features.detach().float().cpu())
            if (start // FEATURE_BATCH_SIZE + 1) % 10 == 0:
                print(f'{model_name}: processed {min(start + FEATURE_BATCH_SIZE, len(selected))}/{len(selected)}')
    del encoder
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    result = torch.cat(chunks).clone()  # Normal tensor for autograd through the head.
    assert result.ndim == 2 and len(result) == len(selected)
    return result

## 3. Train and export one head per encoder

Validation accuracy selects each best head. The new internal test split is evaluated once for model diagnostics; the separate 180-image holdout is reserved for notebook 06. Every run saves its split fingerprint, class mapping, predictions, and metrics to Drive.


In [ ]:
targets = torch.tensor(selected.target.to_numpy(), dtype=torch.long)
positions = {image_id: position for position, image_id in enumerate(selected.image_id)}
SPLIT_POSITIONS = {name: torch.tensor([positions[image_id] for image_id in frame.image_id])
                   for name, frame in MODEL_SPLITS.items()}


def make_loader(features: torch.Tensor, split: str, shuffle: bool) -> DataLoader:
    indices = SPLIT_POSITIONS[split]
    dataset = TensorDataset(features[indices], targets[indices], indices)
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=HEAD_BATCH_SIZE, shuffle=shuffle, generator=generator)


def run_head_epoch(head: nn.Module, loader: DataLoader, optimizer=None):
    training = optimizer is not None
    head.train(training)
    total_loss = total_correct = total_count = 0
    with torch.set_grad_enabled(training):
        for embeddings, labels, _ in loader:
            embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = head(embeddings)
            loss = nn.functional.cross_entropy(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += float(loss.detach()) * len(labels)
            total_correct += int((logits.argmax(1) == labels).sum())
            total_count += len(labels)
    return {'loss': total_loss / total_count, 'accuracy': total_correct / total_count}


def train_and_export(model_name: str) -> Path:
    seed_everything(SEED)
    features = extract_features(model_name)
    head = nn.Sequential(nn.LayerNorm(features.shape[1]), nn.Dropout(0.2),
                         nn.Linear(features.shape[1], len(CLASS_INFO))).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE)
    history = []
    best_score = (-1.0, float('-inf'))
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        train_result = run_head_epoch(head, make_loader(features, 'train', True), optimizer)
        val_result = run_head_epoch(head, make_loader(features, 'val', False))
        row = {'epoch': epoch, 'train_loss': train_result['loss'],
               'train_accuracy': train_result['accuracy'], 'val_loss': val_result['loss'],
               'val_accuracy': val_result['accuracy']}
        history.append(row)
        score = (row['val_accuracy'], -row['val_loss'])
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(head.state_dict())
        print(f"{model_name} epoch {epoch}: train={row['train_accuracy']:.3f} val={row['val_accuracy']:.3f}")
    head.load_state_dict(best_state)
    best_row = max(history, key=lambda row: (row['val_accuracy'], -row['val_loss']))
    test = run_head_epoch(head, make_loader(features, 'test', False))
    truth, predicted, prediction_rows = [], [], []
    head.eval()
    with torch.inference_mode():
        for embeddings, labels, indices in make_loader(features, 'test', False):
            predictions = head(embeddings.to(DEVICE)).argmax(1).cpu().tolist()
            truth.extend(labels.tolist())
            predicted.extend(predictions)
            for position, actual, guess in zip(indices.tolist(), labels.tolist(), predictions, strict=True):
                row = selected.iloc[position]
                prediction_rows.append({
                    'image_id': row.image_id, 'split': row.split, 'source_split': row.source_split,
                    'true_index': actual, 'true_class': CLASS_NAMES[actual],
                    'predicted_index': guess, 'predicted_class': CLASS_NAMES[guess],
                    'correct': actual == guess,
                })
    report = classification_report(
        truth, predicted, labels=list(range(len(CLASS_INFO))), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0
    )
    per_class = [{
        'class_index': index, 'class_name': name,
        'precision': float(report[name]['precision']), 'recall': float(report[name]['recall']),
        'f1_score': float(report[name]['f1-score']), 'support': int(report[name]['support']),
    } for index, name in enumerate(CLASS_NAMES)]
    matrix = confusion_matrix(truth, predicted, labels=list(range(len(CLASS_INFO)))).tolist()
    metrics = {
        'history': history, 'best_epoch': best_row['epoch'],
        'best_val_accuracy': best_row['val_accuracy'], 'best_val_loss': best_row['val_loss'],
        'test': {'accuracy': test['accuracy'], 'loss': test['loss'],
                 'macro_f1': float(report['macro avg']['f1-score']), 'count': len(truth)},
    }
    run_dir = RUNS_DIR / model_name
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, run_dir / 'best_head.pt')
    training = {
        'model_name': model_name, 'hf_dinov3_id': HF_DINOV3_ID,
        'hf_siglip2_id': HF_SIGLIP2_ID, 'seed': SEED,
        'source_version': DATASET_VERSION, 'epochs': EPOCHS,
        'feature_batch_size': FEATURE_BATCH_SIZE, 'head_batch_size': HEAD_BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'freeze_encoder': True, 'split_policy': 'holdout10_then_80_10_10',
    }
    metadata = {
        'version_tag': DATASET_VERSION, 'dataset_name': 'Plant Classifier export',
        'archive_name': ARCHIVE_PATH.name,
        'dataset_fingerprint': DATASET_FINGERPRINT,
        'by_split': {**{name: len(frame) for name, frame in MODEL_SPLITS.items()},
                     'external_test': len(EXTERNAL_SPLIT)},
        'class_count': len(CLASS_INFO), 'original_split_preserved': False,
        'split_policy': 'holdout10_then_80_10_10',
    }
    mapping = {'class_to_idx': {name: i for i, name in enumerate(CLASS_NAMES)},
               'index_to_class': CLASS_NAMES,
               'source_label_indices': {name: source_index for source_index, name, _ in CLASS_INFO}}
    for filename, value in [('config.json', {'training': training}),
                            ('dataset_metadata.json', metadata),
                            ('class_mapping.json', mapping), ('metrics.json', metrics)]:
        (run_dir / filename).write_text(json.dumps(value, indent=2), encoding='utf-8')
    pd.DataFrame(per_class).to_csv(run_dir / 'test_per_class_metrics.csv', index=False)
    pd.DataFrame(prediction_rows).to_csv(run_dir / 'test_predictions.csv', index=False)
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        run_dir / 'test_confusion_matrix.csv'
    )
    del features, head
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print('Saved', run_dir)
    return run_dir

In [ ]:
completed = []
for model_name in MODELS_TO_TRAIN:
    print('Starting', model_name)
    completed.append(train_and_export(model_name))
print('Completed:', [path.name for path in completed])
print('Open notebook 04 for trained-run diagnostics and notebook 05 for Muse Spark comparison.')

Starting efficientnet_b2
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 89.6MB/s]


efficientnet_b2: processed 80/1112
efficientnet_b2: processed 160/1112
efficientnet_b2: processed 240/1112
efficientnet_b2: processed 320/1112
efficientnet_b2: processed 400/1112
efficientnet_b2: processed 480/1112
efficientnet_b2: processed 560/1112
efficientnet_b2: processed 640/1112
efficientnet_b2: processed 720/1112
efficientnet_b2: processed 800/1112
efficientnet_b2: processed 880/1112
efficientnet_b2: processed 960/1112
efficientnet_b2: processed 1040/1112
efficientnet_b2 epoch 1: train=0.427 val=0.667
efficientnet_b2 epoch 2: train=0.803 val=0.757
efficientnet_b2 epoch 3: train=0.907 val=0.802
efficientnet_b2 epoch 4: train=0.957 val=0.820
efficientnet_b2 epoch 5: train=0.970 val=0.838
efficientnet_b2 epoch 6: train=0.987 val=0.829
efficientnet_b2 epoch 7: train=0.992 val=0.838
efficientnet_b2 epoch 8: train=0.996 val=0.847
efficientnet_b2 epoch 9: train=0.997 val=0.847
efficientnet_b2 epoch 10: train=0.999 val=0.856
efficientnet_b2 epoch 11: train=1.000 val=0.838
efficientnet_

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

siglip2: processed 80/1112
siglip2: processed 160/1112
siglip2: processed 240/1112
siglip2: processed 320/1112
siglip2: processed 400/1112
siglip2: processed 480/1112
siglip2: processed 560/1112
siglip2: processed 640/1112
siglip2: processed 720/1112
siglip2: processed 800/1112
siglip2: processed 880/1112
siglip2: processed 960/1112
siglip2: processed 1040/1112
siglip2 epoch 1: train=0.519 val=0.829
siglip2 epoch 2: train=0.874 val=0.892
siglip2 epoch 3: train=0.938 val=0.928
siglip2 epoch 4: train=0.965 val=0.937
siglip2 epoch 5: train=0.966 val=0.955
siglip2 epoch 6: train=0.970 val=0.964
siglip2 epoch 7: train=0.973 val=0.964
siglip2 epoch 8: train=0.983 val=0.964
siglip2 epoch 9: train=0.979 val=0.964
siglip2 epoch 10: train=0.980 val=0.964
siglip2 epoch 11: train=0.984 val=0.964
siglip2 epoch 12: train=0.987 val=0.964
Saved /content/drive/MyDrive/AI/meta-global-hackaton/runs/plant_export_v20261026_2355546_holdout10_seed42/siglip2
Completed: ['efficientnet_b2', 'dinov3', 'siglip2']

## Limits

This holdout is sampled from the original archive, so it measures performance on held-out images from the **same collection**. It cannot establish real-world generalization. Some classes remain small after reserving 10 images, and labels or near-duplicates still need human inspection. Do not use the holdout repeatedly to tune models.
